# Agents in a dev container: Claude Code and Gemini CLI

The notebook is built in two halves:

**Half one — a bare agent.** Install two CLI agents with nothing added, and watch them work on a
real task using only what they ship with. This is the baseline to show where the limit is.

**Half two — an agent with tools.** Hit that limit on purpose, then close it: connect the
Materials Project database over MCP, and write a tool of your own.

Doing it in this order matters. If you install everything at once, you never see what the tools
actually bought you.

| Part | What it covers | Where |
|---|---|---|
| 0 | Prerequisites | your machine |
| 1 | A minimal container: two agents, nothing else | host |
| 2 | Build and sign in to both | VS Code |
| 3 | **Worked example with no extra tools** | container |
| 4 | Where the bare agent stops | container |
| 5 | Add the Materials Project over MCP | container |
| 6 | Project instructions and permissions | container |
| 7 | Check what the agent can actually reach | container |
| 8 | Write your own tool | container |
| 9 | Verify end to end | container |
| 10 | Make it reproducible | container |

**How to use it.** Markdown cells hold files to create and commands to type in a terminal. Code
cells are *verification checks* — run them with the container's Python kernel to confirm each step
worked before moving on.

---
## Terminology

The notebook uses vocabulary from software engineering that is not standard in computational
materials science. The terms below are defined in the order in which they become relevant.

### Execution environment

**Host.** The physical or virtual machine on which you work directly — your laptop or
workstation. Every operation labelled *host* in this notebook runs on that machine, outside any
isolated environment.

**Container.** An isolated user-space environment that runs on the host's operating-system kernel
but has its own file system, installed software and process space. Unlike a virtual machine it
does not emulate hardware, so it starts in seconds and has negligible overhead. Software installed
inside a container is not visible on the host, and removing the container leaves the host
unchanged. The closest HPC analogue is a `module` environment on SDumont, with the difference
that a container isolates *everything*, not just the search paths.

**Image.** The read-only template from which a container is instantiated: an operating system plus
a specified set of software, built in layers. Image : container is analogous to input file :
running job — the same image yields identical containers on any machine.

**Docker.** The runtime that builds images and runs containers. It is the only piece of
container software installed on the host.

**Dev container.** A container defined by a JSON specification (`.devcontainer/devcontainer.json`)
that VS Code builds and then attaches to, so that the editor, terminal and Jupyter kernel all
execute inside it. The specification file is the reproducible record of the environment: commit it
and any collaborator obtains the same setup.

**Volume.** A storage area managed by Docker that persists independently of any container. The
container's own file system is discarded when it is rebuilt; data written to a volume survives.
Here volumes hold the agents' login credentials.

**Environment variable.** A named string passed to every process started in a shell
(e.g. `MP_API_KEY`). It is the conventional way to supply secrets and configuration without
writing them into files that may be committed.

**Kernel (Jupyter).** The process that executes a notebook's code cells. In this notebook the
kernel runs inside the container, so the verification cells test the container, not the host.

### Agents and tools

**Large language model (LLM).** A neural network trained to predict text. On its own it maps an
input text to an output text; it cannot read your files, execute code or query a database.

**Agent.** A program that places an LLM inside a loop: the model proposes an action, the program
executes it, the result is returned to the model, and the cycle repeats until the task is complete
or a human intervenes. The distinction from a chat interface is that the model's output is
*executed* and the model observes the real result — including error messages.

**CLI (command-line interface).** A program operated by typed commands in a terminal rather than
through a graphical interface. Both agents here — **Claude Code** (Anthropic) and **Gemini CLI**
(Google) — are CLI agents, which is what allows them to run inside a container or over SSH on a
cluster.

**Tool / tool call.** A *tool* is a function the agent is permitted to invoke: read a file, write
a file, execute a shell command, query a database. A *tool call* is one such invocation. Every
factual claim an agent makes should be traceable to a tool call; a claim with no tool call behind
it comes from the model's training data and is unverified.

**MCP (Model Context Protocol).** An open standard that specifies how an agent discovers and calls
tools provided by an external program. An **MCP server** is a program that exposes a set of tools
under this protocol — for example, `search` and `fetch` over the Materials Project database. The
agent acts as the **client**. Because the protocol is shared, one server works with any compliant
agent: the Materials Project server configured in Part 5 can be registered with Claude Code,
Gemini CLI or Codex CLI without modification.

**stdio transport.** The mode in which the agent launches the MCP server as a child process and
exchanges messages with it over standard input and output. It is the reason a server started by
hand in a terminal appears to hang: it is waiting for input on stdin.

### Services and credentials

**API (application programming interface).** A defined set of requests that one program can send
to another. The Materials Project exposes its database through a web API; an MCP server is a
thin layer that turns that API into agent tools.

**API key.** A secret string that authenticates your requests to an API. It identifies you, and
it must never be committed to a repository or pasted into a shared notebook.

**Materials Project.** An open database of DFT-computed properties for inorganic compounds
(structures, formation energies, energy above the convex hull, band structures), accessible
through its web API with a free key.

**Node.js / npm; Python / pip.** Language runtimes and their package managers. Gemini CLI is
distributed through npm; the Materials Project MCP server and `pymatgen` through pip.

---
## Part 0 — Prerequisites, step by step

All on **your own machine**, before any container exists.

### 0.1 Install Docker

- **macOS / Windows/ Linux:** install [Docker Desktop](https://www.docker.com/products/docker-desktop/) and launch it. Wait until it reports running.

Verify:
```bash
docker --version
docker run --rm hello-world
```

### 0.2 Install VS Code and the Dev Containers extension

Install [Visual Studio Code](https://code.visualstudio.com/), then from the Extensions sidebar
install **Dev Containers** by Microsoft (`ms-vscode-remote.remote-containers`).

This is the only extension you install on the host. Everything else goes inside the container.

### 0.3 Accounts

- **Claude:** a Pro, Max, Team or Console account. You sign in from inside the container in Part 2.
- **Gemini:** a Google account is enough — Gemini CLI signs in with it directly. An API key from
  [AI Studio](https://aistudio.google.com/apikey) is an alternative.

### 0.4 Materials Project API key

You don't need this until Part 5, but set it now — `containerEnv` is read when the container is
*created*, so adding it later would mean another rebuild.

1. Register at [materialsproject.org](https://materialsproject.org) and copy the key from your dashboard.
2. Put it in your host shell profile, `~/.bashrc`:
   ```bash
   export MP_API_KEY=your_key_here
   ```
3. Open a new terminal and check: `echo $MP_API_KEY`

**Windows with WSL2:** put the export in the WSL shell's `~/.bashrc` and launch VS Code from
inside WSL. **Windows without WSL:** `setx MP_API_KEY "your_key_here"`, then reopen every terminal
and VS Code window.

> **One problem.** `${localEnv:MP_API_KEY}` reads the environment **VS Code
> itself was launched with**, not your shell's. On macOS an app started from the Dock never reads
> `~/.zshrc`, so the variable is empty however good your terminal looks.
>
> Launch VS Code from a terminal with `code .`, or fully quit it (`Cmd+Q`, not just closing the
> window) and reopen after setting the variable.
>
> It fails *silently* — the variable becomes an empty string, the container builds fine, and you
> find out in Part 5 when database calls return authentication errors.

### 0.5 Create the project folder

```bash
mkdir ~/agents-demo
cd ~/agents-demo
code .
```

Use `code .` rather than the VS Code UI, so the window inherits your shell environment.

### 0.6 Host check — all four must pass

```bash
docker --version
docker ps
echo $MP_API_KEY
code --list-extensions | grep remote-containers
```

A blank `MP_API_KEY` means go back to 0.4. Don't continue — everything will appear to work until
Part 5.

---
## Part 1 — A minimal container: two agents, nothing else

Just the two agents installation.

Create `.devcontainer/devcontainer.json`:

```json
{
  "name": "Claude Code Environment",
  "image": "mcr.microsoft.com/devcontainers/python:3.12",

  "features": {
    "ghcr.io/devcontainers/features/node:1": { "version": "22" },
    "ghcr.io/anthropics/devcontainer-features/claude-code:1": {}
  },

  "remoteUser": "vscode",

  "containerEnv": {
    "MP_API_KEY": "${localEnv:MP_API_KEY}",
    "CLAUDE_CONFIG_DIR": "/home/vscode/.claude"
  },

  "mounts": [
    "source=claude-config-${devcontainerId},target=/home/vscode/.claude,type=volume",
    "source=gemini-config-${devcontainerId},target=/home/vscode/.gemini,type=volume"
  ],

  "postCreateCommand": "sudo chown -R vscode:vscode /home/vscode/.claude /home/vscode/.gemini && npm install -g @google/gemini-cli",

  "customizations": {
    "vscode": {
      "extensions": [
        "ms-python.python",
        "ms-toolsai.jupyter",
        "pkief.material-icon-theme"
      ]
    }
  }
}
```

### How VS Code uses this file

When you ask VS Code to open the folder in a container (Part 2), it goes through two stages:

1. **Build.** VS Code starts from a ready-made Linux system with Python 3.12 (`image`) and installs the software
   listed under `features` on top of it. The result is saved on your machine as a new image. This
   is slow the first time, but Docker keeps the result, so later builds reuse it instead of
   installing everything again.
2. **Create and start.** VS Code starts a container from that image, applies the settings
   (environment variables, storage, user), and finally runs the command in `postCreateCommand`.

Whenever you change `devcontainer.json`, VS Code must **rebuild**: it throws away the running
container and repeats these steps. Anything saved only inside the old container is lost. Several
choices below exist to protect you from that.

### What each entry does

| Entry | What it means |
|---|---|
| `name` | The label shown in VS Code. Cosmetic. |
| `image` | The starting system: a Debian Linux with Python 3.12 already installed, prepared by Microsoft for dev containers. It contains Python and `pip`, but no Node.js and no agents. |
| `features` | Ready-made installation packages. Each line installs one piece of software during the build: Node.js 22 and Claude Code. Python needs no feature, because the image already provides it. |
| `remoteUser` | The user account you work as inside the container. `vscode` is an ordinary user, not the administrator (`root`). |
| `containerEnv` | Environment variables set inside the container (see *Terminology*). |
| `mounts` | Storage connected to the container from outside it — here, two Docker volumes. |
| `postCreateCommand` | A shell command run once, right after each container is created. |
| `customizations` | VS Code extensions to install inside the container: Python, Jupyter, and an icon theme. |

### Why the two agents are installed differently

**Claude Code is installed as a `feature`.** Anthropic publishes an official feature for dev
containers, so a single line in `features` is enough. Because features are installed during the
build, the result is saved together with the image and not repeated on every rebuild. The feature
also installs the Claude Code extension for VS Code.

**Gemini CLI is installed in `postCreateCommand`.** Google does not publish a feature, so we
install it with an ordinary command: `npm install -g @google/gemini-cli`. (`npm` is the installer
for Node.js programs, much like `pip` for Python.) The disadvantage is that this command runs again
every time the container is recreated, so each rebuild downloads Gemini CLI again.

Both approaches work. Seeing them side by side shows the trade-off: a feature is saved and reused;
a command is repeated.

### Why the Python image, and why Node.js is still added

**The image already provides Python.** Parts 3 to 8 run Python scripts, Jupyter cells and a
Python-based MCP server, so we start from Microsoft's Python 3.12 image instead of a bare Linux
system. Python and `pip` are then part of the starting point and need no extra installation.

**Node.js is required by both agents.** Claude Code and Gemini CLI are Node.js programs, and the
Python image does not include Node.js or `npm`. Without the Node feature, the `npm install` line
in `postCreateCommand` would fail immediately.

### Why the two volumes

The agents save your login in your home folder: Claude Code in `~/.claude`, Gemini CLI in
`~/.gemini`. But the home folder belongs to the container, and a rebuild deletes it — so without
extra care you would have to sign in to both agents again after every change to this file.

A **volume** solves this. It is storage kept by Docker *outside* the container, and the `mounts`
entry attaches it at the given folder. Files written there survive a rebuild. We use one volume for
each agent. `${devcontainerId}` in the name is replaced by an identifier unique to this project,
so two different projects do not share the same login.

**Why `CLAUDE_CONFIG_DIR` is also needed.** Claude Code stores most of its settings in
`~/.claude/`, but it also writes one file, `~/.claude.json`, *next to* that folder rather than
inside it. That file would sit outside the volume and be lost on rebuild. Setting
`CLAUDE_CONFIG_DIR=/home/vscode/.claude` tells Claude Code to keep everything inside the folder
that is saved.

### Why the `chown` command

When Docker creates a volume, the folder belongs to the administrator account, `root`. You work as
the user `vscode`, which is not allowed to write there. The first thing each agent tries to save —
your login — would then fail with `Permission denied`.

`sudo chown -R vscode:vscode <folders>` changes the owner of those folders to `vscode`. (`sudo`
runs a command with administrator rights; `chown` means *change owner*; `-R` applies it to
everything inside.) It is placed at the start of `postCreateCommand` so it runs before anything
else.

### Why `MP_API_KEY` is already here

Nothing uses the Materials Project key until Part 5. It is added now because environment variables
in `containerEnv` are fixed when the container is created. `${localEnv:MP_API_KEY}` means *copy the
value of `MP_API_KEY` from the host*. If you added this line later, you would need another
rebuild.

---
## Part 2 — Build and sign in to both

1. Command Palette (`Ctrl+Shift+P` / `Cmd+Shift+P`) → **Dev Containers: Reopen in Container**.
2. Wait. The first build takes several minutes.
3. Open a terminal inside the container (`` Ctrl+` ``).

**Claude Code:**
```bash
claude
```
Follow the browser sign-in. If the browser completes but the terminal never returns, copy the code
shown in the browser and paste it at the `Paste code here if prompted` prompt — that happens when
port forwarding doesn't route the localhost callback.

**Gemini CLI:**
```bash
gemini
```
Choose *Sign in with Google* and follow the same flow. If the callback doesn't reach the container,
use an API key instead:
```bash
export GEMINI_API_KEY=your_key_from_aistudio
```

Both store their state in the volumes you mounted, so this is a one-time step.

Run the cell below to confirm the environment.

In [ ]:
import os, shutil, sys
from pathlib import Path

print("user   :", os.environ.get("USER", "?"))
print("home   :", Path.home())
print("cwd    :", Path.cwd())
print("python :", sys.version.split()[0])
print()
for tool in ("claude", "gemini", "node", "npm"):
    print(f"{tool:8}:", shutil.which(tool) or "NOT FOUND")
print()
print("MP_API_KEY       :", "set" if os.environ.get("MP_API_KEY") else "MISSING (see Part 0.4)")
print("CLAUDE_CONFIG_DIR:", os.environ.get("CLAUDE_CONFIG_DIR", "not set"))

In [ ]:
# Both config directories must be writable by you, not by root.
import os
from pathlib import Path

for d in (Path.home() / ".claude", Path.home() / ".gemini"):
    if not d.exists():
        print(f"{d}: does not exist yet (created on first sign-in)")
        continue
    probe = d / ".write-test"
    try:
        probe.touch(); probe.unlink()
        print(f"{d}: writable")
    except PermissionError:
        print(f"{d}: PERMISSION DENIED -> sudo chown -R vscode:vscode {d}")

---
## Part 3 — Worked example, with no extra tools

This is the baseline. Neither agent has a database connection, a materials library, or anything
you wrote. They have what they ship with: read a file, write a file, run a command.

First, create a small structure file to work on. Run the cell below — it writes an **idealised
rock-salt NiO prototype** with a = 4.17 Å. This is a teaching file, not an experimental
refinement, and it's chosen because the answer is arithmetic you can check in your head: in
rock-salt, the nearest-neighbour Ni–O distance is exactly a/2.

In [ ]:
from pathlib import Path

POSCAR = """Idealised rock-salt NiO prototype (teaching file, a = 4.17 A)
1.0
   4.1700000000    0.0000000000    0.0000000000
   0.0000000000    4.1700000000    0.0000000000
   0.0000000000    0.0000000000    4.1700000000
Ni O
4 4
Direct
   0.000000  0.000000  0.000000
   0.000000  0.500000  0.500000
   0.500000  0.000000  0.500000
   0.500000  0.500000  0.000000
   0.500000  0.500000  0.500000
   0.500000  0.000000  0.000000
   0.000000  0.500000  0.000000
   0.000000  0.000000  0.500000
"""

Path("structures").mkdir(exist_ok=True)
p = Path("structures/NiO_prototype_POSCAR")
p.write_text(POSCAR)
print("wrote", p)
print()
print("Expected nearest-neighbour Ni-O distance = a/2 =", 4.17 / 2, "A")
print("Keep that number. It is how you check the agent's answer.")

Now give **the same prompt to both agents**, in a terminal.

```
Read structures/NiO_prototype_POSCAR. Write a short Python script that computes
every Ni-O distance shorter than 3 A, including periodic images, and run it.
Report the nearest-neighbour distance and how many neighbours are at that
distance. Use only the standard library. Don't install anything.
```

Run it once with `claude`, then once with `gemini`, in the same folder.

**What to watch — not the prose, the tool calls.** Both will read the file, write a script, run it,
and read the output. If the script errors, watch what happens next: the agent sees the real
traceback and corrects itself. That feedback loop is the entire difference between an agent and a
chat window.

**What the answer should be.** Nearest neighbour 2.085 Å, with 6 neighbours at that distance —
octahedral coordination. If an agent reports something else, it made an error you can catch
instantly, which is exactly why this example was chosen.

Compare the two runs: how many steps each took, whether either hit an error and recovered, and
whether the reported number is right.

In [ ]:
# What the agents left behind.
from pathlib import Path

for f in sorted(Path(".").glob("*.py")) + sorted(Path("structures").glob("*")):
    print(f"{f.stat().st_size:>7} B  {f}")
print("\nRead any script they wrote. You are allowed to disagree with it.")

---
## Part 4 — Where the bare agent stops

Now ask something the local files cannot answer. Same terminal, either agent:

```
What is the energy above hull of La3Ni2O7 in the Materials Project database,
and what is its Materials Project ID? Only answer from a source you can
actually query. If you cannot verify it, say so.
```

One of three things happens, and all three are instructive:

1. **It says it cannot check.** Correct, and the behaviour you want.
2. **It answers from memory.** Plausible-looking, possibly right, entirely unverifiable — you have
   no way to tell which without going to look yourself.
3. **It searches the web** and reports what a page said. Better, but that is a web page, not the
   database.

None of these is the database. The model has no connection to Materials Project, and no amount of
prompting creates one.

**This is the gap tools fill**, and it's worth sitting with for a moment before closing it. The
limit isn't intelligence — it's reach. Everything from here on is about giving the agent reach,
and about controlling what that reach includes.

---
## Part 5 — Add the Materials Project over MCP

**MCP** (Model Context Protocol) is an open standard for exposing tools and data to any agent.
The Materials Project maintains a server; you connect to it, and its tools appear in your session.
The same server works in Claude Code, Gemini CLI and Codex CLI — you are not locked to one vendor.

### 5.1 Install the server

Create `.devcontainer/setup-mp-mcp.sh`:

```bash
#!/usr/bin/env bash
set -euo pipefail
HERE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"

SRC="$HOME/mp-mcp"
[ -d "$SRC" ] || git clone https://github.com/esoteric-ephemera/mp_api.git "$SRC"
cd "$SRC"

# The MCP server lives on the auto-dependency-upgrades branch, NOT on main,
# and this repository has no tags at all. Pinned to a commit deliberately.
git checkout 553f725af71d48197424807b02ae8c80bc32d640

python3 -m venv "$SRC/.venv"
"$SRC/.venv/bin/pip" install --upgrade pip
"$SRC/.venv/bin/pip" install -e '.[mcp]' -c "$HERE/mp-mcp-constraints.txt"

echo "MCP server interpreter: $SRC/.venv/bin/python"
```

And `.devcontainer/mp-mcp-constraints.txt`:

```
emmet-core==0.86.4
```

Run it once by hand: `bash .devcontainer/setup-mp-mcp.sh`

Then add it to `postCreateCommand` in `devcontainer.json` so rebuilds get it automatically:

```json
"postCreateCommand": "sudo chown -R vscode:vscode /home/vscode/.claude /home/vscode/.gemini && npm install -g @google/gemini-cli && bash .devcontainer/setup-mp-mcp.sh",
```

### Two pins, both learned the hard way

**The commit SHA.** The Materials Project documentation says `git checkout v0.46.0`. That tag does
not exist in this fork — it has no tags at all, only two branches. And `main` has no `mp_api/mcp/`
directory and no `mcp` extra, so simply dropping the checkout fails one line later instead. Pin
the SHA rather than tracking the branch: the name says a bot pushes to it, and it can be
force-pushed or deleted without warning.

**`emmet-core`.** The package requires `emmet-core>=0.86.3rc0` with no upper bound, so pip takes
the newest. Version **0.87.0 moved `BSPathType`** from `emmet/core/electronic_structure.py` to
`emmet/core/band_theory.py`, breaking the import chain. `0.86.4` is the last version with it in
the old place.

### 5.2 Register the server

Create `.mcp.json` in the **project root**:

```json
{
  "mcpServers": {
    "materials-project": {
      "command": "/home/vscode/mp-mcp/.venv/bin/python",
      "args": ["-m", "mp_api.mcp.server"],
      "env": {
        "MP_API_KEY": "${MP_API_KEY}"
      }
    }
  }
}
```

**The absolute interpreter path matters.** Claude Code launches the server as a separate process
and knows nothing about an activated virtual environment. A bare `python` resolves to whatever is
first on `PATH` in *its* environment, `mp_api` isn't importable, and the server fails silently.

**`${MP_API_KEY}` is expanded by Claude Code**, so the literal key never enters the file.
**`${CLAUDE_PROJECT_DIR}` does not work here** — that variable is for hooks. Use absolute paths.

In [ ]:
# Check the config parses and the interpreter exists, before restarting anything.
import json
from pathlib import Path

cfg = Path.cwd() / ".mcp.json"
try:
    data = json.loads(cfg.read_text())
except FileNotFoundError:
    raise SystemExit("no .mcp.json here - are you in the project root?")
except json.JSONDecodeError as exc:
    raise SystemExit(f"INVALID JSON: {exc}\nA malformed .mcp.json fails silently. Fix this first.")

for name, srv in data["mcpServers"].items():
    exe = Path(srv["command"])
    print(name)
    print("  command :", exe, "->", "exists" if exe.exists() else "MISSING")
    print("  args    :", srv.get("args"))
    print("  env     :", list(srv.get("env", {})))

In [ ]:
# Can that exact interpreter import the server module?
import subprocess

PY = "/home/vscode/mp-mcp/.venv/bin/python"
r = subprocess.run([PY, "-c", "import mp_api.mcp.server; print('import ok')"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr[-1500:])
print("\nIf you see 'cannot import name BSPathType', the emmet-core pin did not take:")
print("  $HOME/mp-mcp/.venv/bin/pip install 'emmet-core==0.86.4'")

The real test is launching the server exactly as the agent would. **In a terminal**, not here:

```bash
MP_API_KEY="$MP_API_KEY" $HOME/mp-mcp/.venv/bin/python -m mp_api.mcp.server
```

A FastMCP banner and then silence means it works — it is waiting on stdin. Press **Ctrl-D** to
exit (Ctrl-C often won't, because the process is blocked reading stdin).

Restart Claude Code — MCP connections are made at launch — and run `/mcp`. You want
`materials-project ✔ connected` with two tools, `search` and `fetch`.

**Now re-run the Part 4 question.** Same prompt, different answer: an MP ID and a number from the
database. That contrast is the whole point of the notebook.

---
## Part 6 — Project instructions and permissions

Two files, two different jobs.

### `CLAUDE.md` — what the agent *should* do

Read automatically at the start of every session, so your conventions stop living in prompts.

```markdown
# CLAUDE.md

Demo and teaching environment for a seminar on AI agents in computational
materials science. Small, fast calculations only.

- Do not modify `.devcontainer/`, `.mcp.json`, or `.claude/` without asking.

## Project conventions

### Structures
- Structures go in `./structures`, as POSCAR files.

### DFT settings
- VASP 6.x, PBEsol, PAW_PBE 54 pseudopotentials.
- Default ENCUT 520 eV; default k-spacing 0.03 2π/Å.
- Never change the functional or pseudopotential set without asking.
- Convergence is not assumed — state what was checked.

### Job submission
- Never submit anything to a queue. Never run `sbatch` or `srun`.
- When a calculation needs submitting, show me the script instead.

### Working style
- Propose a plan and wait for approval before editing files.
- Make one logical change at a time.
- Do not fabricate numerical results, outputs, database records, or commands.
- Clearly distinguish database, literature, calculated, and estimated values.

### Materials Project
- Use the `materials-project` MCP server for database queries.
- Report the Materials Project ID for database-derived results.
- Never invent Materials Project IDs or properties.
```

Replace the DFT settings with your group's actual conventions.

Gemini CLI reads `GEMINI.md` for the same purpose — the same content works.

### `.claude/settings.json` — what the agent *can* do

```json
{
  "$schema": "https://json.schemastore.org/claude-code-settings.json",
  "permissions": {
    "disableBypassPermissionsMode": "disable",
    "blockReadsOutsideWorkingDirectories": true,
    "deny": [
      "Read(./.env)",
      "Read(~/.ssh/**)",
      "Bash(sbatch:*)",
      "Bash(srun:*)",
      "Bash(scancel:*)"
    ]
  }
}
```

**The distinction is the point.** `CLAUDE.md` is a *request* — followed because the agent is
cooperative. `deny` rules are checked by the permission system before a command runs, and no
prompt gets past them. Use both: the first so it doesn't try, the second so it can't.

The value for those flags is the string `"disable"`, not a boolean.

`.claude/settings.json` is committed and shared. `.claude/settings.local.json` is your personal
file in the same folder, kept out of git, and it accumulates your standing approvals.

In [ ]:
import json
from pathlib import Path

for p in (Path("CLAUDE.md"), Path(".claude/settings.json")):
    if not p.exists():
        print(f"{p}: MISSING")
        continue
    print(f"{p}: {p.stat().st_size} B")
    if p.suffix == ".json":
        try:
            cfg = json.loads(p.read_text())
            print("   parses OK; deny:", cfg.get("permissions", {}).get("deny", []))
        except json.JSONDecodeError as exc:
            print("   INVALID JSON:", exc)

---
## Part 7 — Check what the agent can actually reach

Run `/mcp` and read the list carefully.

If your Claude account has **claude.ai connectors** enabled — Gmail, Google Drive, Google
Calendar — Claude Code fetches them from your account automatically and they appear here. Nothing
in the container asked: the Google OAuth grant happened earlier on claude.ai, and this inherits it.

Worth knowing before you put a session on a projector. Three connectors is roughly forty tools
reaching a personal email account, and their definitions sit in the model's context.

To clean up the demo without touching your everyday Claude setup:

- **Per project:** toggle each connector off in the `/mcp` screen. Recorded in `~/.claude.json`,
  which is on your persisted volume, so it survives rebuilds.
- **For anyone who clones the repo:** `"disableClaudeAiConnectors": true` in `.claude/settings.json`.
- **One session:** `ENABLE_CLAUDEAI_MCP_SERVERS=false claude`.

To disconnect from your account entirely, that's claude.ai under *Customize → Connectors*, plus
your Google account's third-party access settings.

This is also the most concrete version of a point worth making out loud: *know what your agent can
reach*. It is easy to be wrong about, on your own machine, without anything having gone wrong.

---
## Part 8 — Write your own tool

The Materials Project server returns metadata only — no atomic coordinates. To go from a database
hit to a VASP input set you need tools of your own. A tool is an ordinary Python function; the
docstring is the interface the model reads.

Create `tools/vasp_inputs_server.py`:

```python
import os
from pathlib import Path

from fastmcp import FastMCP
from mp_api.client import MPRester
from pymatgen.core import Structure
from pymatgen.io.vasp.sets import MPRelaxSet

mcp = FastMCP("VASP Input Builder")


@mcp.tool
def get_mp_structure(mp_id: str, out_dir: str = "structures") -> dict:
    """Download a crystal structure from the Materials Project as a POSCAR.

    Use this when you need atomic coordinates — the materials-project
    server only returns summary metadata, not structures.

    Args:
        mp_id: Materials Project ID, e.g. "mp-18926"
        out_dir: directory to write the POSCAR into
    """
    with MPRester(os.environ["MP_API_KEY"]) as mpr:
        struct = mpr.get_structure_by_material_id(mp_id)
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    path = out / f"{mp_id}_POSCAR"
    struct.to(filename=str(path), fmt="poscar")
    return {"path": str(path),
            "formula": struct.composition.reduced_formula,
            "spacegroup": struct.get_space_group_info()[0],
            "n_sites": len(struct),
            "lattice_abc": [round(x, 4) for x in struct.lattice.abc]}


@mcp.tool
def write_mp_relax_inputs(poscar_path: str, out_dir: str,
                          encut: float | None = None) -> dict:
    """Write MP-standard VASP relaxation inputs for a structure.

    Uses pymatgen's MPRelaxSet — the same parameter set the Materials
    Project uses. Writes INCAR, KPOINTS, POSCAR and POTCAR.spec.
    POTCAR data is never written, and nothing is submitted to a queue.

    Args:
        poscar_path: path to a POSCAR or CIF
        out_dir: directory to write the input set into
        encut: optional plane-wave cutoff override in eV
    """
    struct = Structure.from_file(poscar_path)
    vis = MPRelaxSet(struct, user_incar_settings={"ENCUT": encut} if encut else {})
    out = Path(out_dir)
    vis.write_input(str(out), potcar_spec=True)
    return {"dir": str(out),
            "files": sorted(p.name for p in out.iterdir()),
            "INCAR": (out / "INCAR").read_text()}


if __name__ == "__main__":
    mcp.run()
```

**Every tool needs `@mcp.tool`.** Without it the function is defined but never registered, and the
agent behaves exactly as if it did not exist. That is a genuinely confusing hour to lose.

**POTCARs are VASP-licensed and not distributable.** `potcar_spec=True` writes a `POTCAR.spec`
listing which pseudopotentials would be used, with none of the licensed data. Never put POTCARs in
a container image or a repository.

Register it alongside the first server in `.mcp.json`:

```json
"vasp-inputs": {
  "command": "/home/vscode/mp-mcp/.venv/bin/python",
  "args": ["/workspaces/<your-folder>/tools/vasp_inputs_server.py"],
  "env": { "MP_API_KEY": "${MP_API_KEY}" }
}
```

Mind the comma after the previous server's closing brace.

In [ ]:
# Confirm every tool carries the @mcp.tool decorator, without running the server.
import ast
from pathlib import Path

src = Path("tools/vasp_inputs_server.py")
if not src.exists():
    print("tools/vasp_inputs_server.py not found")
else:
    tree = ast.parse(src.read_text())
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            decs = [ast.unparse(d) for d in node.decorator_list]
            ok = any("mcp.tool" in d for d in decs)
            print(("registered  " if ok else "NOT A TOOL  ") + node.name + "()")
    print()
    print("A function without @mcp.tool is invisible to the agent.")

---
## Part 9 — Verify end to end

Restart Claude Code. `/mcp` should show **two** servers: `materials-project` with `search` and
`fetch`, and `vasp-inputs` with `get_mp_structure` and `write_mp_relax_inputs`.

**Test 1 — does the key reach the server?**

```
What phases exist in the La-Ni-O system? Give me formula, material ID,
space group and energy above hull, sorted by energy above hull.
```

A table proves it. An auth error appears *here*, not at startup, which is why this step is not
optional.

**Test 2 — do the tools compose?**

```
Fetch La3Ni2O7 from Materials Project, save the structure to ./structures,
and build an MP-standard VASP relaxation input set for it. Show me the INCAR.
Don't submit anything.
```

Three tool calls chained from one sentence, ending in files on disk. Nobody wrote a
"Materials-Project-to-VASP-input" function — the agent composed two tools it was never told to
combine.

In [ ]:
from pathlib import Path

for root in ("structures", "relax"):
    p = Path(root)
    if not p.exists():
        print(f"{root}/  (nothing yet)")
        continue
    print(f"{root}/")
    for f in sorted(p.rglob("*")):
        if f.is_file():
            print(f"  {f.stat().st_size:>8} B  {f.relative_to(p)}")

In [ ]:
# Read the generated INCAR yourself. Never take the summary's word for it.
from pathlib import Path

incars = sorted(Path("relax").rglob("INCAR")) if Path("relax").exists() else []
if not incars:
    print("no INCAR yet")
else:
    text = incars[0].read_text()
    print(incars[0], "\n")
    print(text)
    tags = [ln.split("=")[0].strip() for ln in text.splitlines() if "=" in ln]
    dupes = {t for t in tags if tags.count(t) > 1}
    print("\nduplicate tags:", dupes or "none")

**Read that INCAR critically.** For a *metallic* system, MPRelaxSet's `ISMEAR = -5`
(tetrahedron) combined with `ISIF = 3` is the combination usually warned about: tetrahedron
integration is accurate for energies but less reliable for forces and stress, and `ISIF = 3`
relaxes the cell from the stress tensor. pymatgen ships `MPMetalRelaxSet` for exactly this case.
For an *insulating* halide perovskite the same default is perfectly reasonable.

Same tag, same tool, same agent — right in one case and wrong in the other, and only domain
knowledge tells them apart. Nothing there is a hallucination. That is a harder failure mode to
talk about than invented facts, and a more important one.

---
## Part 10 — Make it reproducible

Freeze the dependency set once everything works. This environment works because of a specific
version alignment; capture it while it is working.

```bash
$HOME/mp-mcp/.venv/bin/pip freeze --exclude-editable \
  | grep -v '^mp-api' > .devcontainer/mp-mcp-constraints.txt
```

`--exclude-editable` and the `grep` both matter: `pip freeze` writes a line for the editable
`mp_api` install itself, which a constraints file cannot contain.

Then **rebuild the container from scratch** — Command Palette → *Dev Containers: Rebuild
Container* — and re-run Part 9. Green with no manual steps means it is reproducible.

**For a truly cold test**, open a *different empty folder* and work through this notebook from
Part 0. A different workspace gets a different `${devcontainerId}`, so different volumes, and your
working setup is untouched. Rebuilding proves your config works; starting from an empty folder
proves your *instructions* work.

In [ ]:
from pathlib import Path

c = Path(".devcontainer/mp-mcp-constraints.txt")
if not c.exists():
    print("constraints file not created yet")
else:
    lines = [ln for ln in c.read_text().splitlines() if ln.strip()]
    print(f"{len(lines)} pinned packages")
    bad = [ln for ln in lines if ln.startswith("-e") or "@" in ln or "mp-api" in ln]
    print("problem lines:", bad or "none")
    print("emmet:", [ln for ln in lines if ln.lower().startswith("emmet")])

---
## Troubleshooting — errors that actually happened

| Error | Cause | Fix |
|---|---|---|
| `MP_API_KEY` empty in the container although the host shell has it | VS Code launched from the Dock/Finder never read your shell profile; `${localEnv:...}` became an empty string | Fully quit VS Code, relaunch with `code .` from a terminal |
| `docker: permission denied` on Linux | Not in the `docker` group yet | `sudo usermod -aG docker $USER`, log out and back in |
| `npm: command not found` in `postCreateCommand` | No Node feature; the base image has no npm | Add `ghcr.io/devcontainers/features/node:1` |
| `touch: cannot touch '/home/vscode/.claude/...': Permission denied` | Docker created the named volume as root | `sudo chown -R vscode:vscode`, and keep it in `postCreateCommand` |
| `error: pathspec 'v0.46.0' did not match any file(s) known to git` | The fork has no tags. The MP docs are wrong. | `git checkout 553f725af71d48197424807b02ae8c80bc32d640` |
| `pip install -e '.[mcp]'` finds no `mcp` extra | You are on `main`, which has no MCP code | Same fix — check out the pinned commit |
| `ImportError: cannot import name 'BSPathType' from 'emmet.core.electronic_structure'` | `emmet-core` ≥ 0.87 moved it to `band_theory.py`; the dependency is unpinned | `pip install "emmet-core==0.86.4"` and pin it |
| `postCreateCommand ... failed with exit code 1` | The script aborted under `set -e`; later steps never ran | Fix the failing line, then re-run the script by hand |
| `materials-project · ✘ failed` in `/mcp` | Usually the venv is missing or the interpreter path is wrong | `claude --debug`, then launch the server by hand for the traceback |
| Server starts but tool calls return auth errors | `MP_API_KEY` empty inside the container | `echo $MP_API_KEY`; check `${localEnv:MP_API_KEY}` and Part 0.4 |
| `variable claude_project_dir is not defined` | `${CLAUDE_PROJECT_DIR}` is for hooks, not `.mcp.json` | Use an absolute path |
| Terminal frozen after Ctrl-C on the MCP server | The process is blocked reading stdin | **Ctrl-D**, or `pkill -f mp_api.mcp.server` |
| A tool you wrote never appears in `/mcp` | Missing `@mcp.tool` decorator | Add it; check with the cell in Part 8 |
| `Unable to import fastmcp` squiggles in VS Code | The editor is using a different interpreter than the server | *Python: Select Interpreter* → `/home/vscode/mp-mcp/.venv/bin/python`. Cosmetic only. |
| Command Palette finds no `Python:` commands | The Python extension is not installed in the container | Extensions sidebar → Python → **Install in Dev Container** |
| Gmail / Drive / Calendar appear in `/mcp` unasked | claude.ai connectors are fetched from your account | Toggle off in `/mcp`, or `disableClaudeAiConnectors` |
| `Error: Connection closed` from an MCP server mid-run | Server died or timed out under rapid sequential calls | `claude --debug` to see why; retry the failed queries |

### The pattern worth noticing

Almost none of the hours this took went on anything to do with agents. A stale tag in someone's
documentation, a branch that isn't `main`, an unpinned transitive dependency, a volume ownership
default. The hard part of agentic tooling in computational materials science is still ordinary
software plumbing — which is itself worth saying to anyone you are teaching.